# मायक्रोसॉफ्ट एजंट फ्रेमवर्कसह ह्यूमन-इन-द-लूप वर्कफ्लो

## 🎯 शिकण्याचे उद्दिष्टे

या नोटबुकमध्ये, आपण मायक्रोसॉफ्ट एजंट फ्रेमवर्कच्या `RequestInfoExecutor` वापरून **ह्यूमन-इन-द-लूप** वर्कफ्लो कसे अंमलात आणायचे ते शिकाल. हा शक्तिशाली पॅटर्न AI वर्कफ्लो थांबवून मानवी इनपुट गोळा करण्यास परवानगी देतो, ज्यामुळे तुमचे एजंट परस्परसंवादी होतात आणि मानवी व्यक्तींना महत्त्वाच्या निर्णयांवर नियंत्रण मिळते.

## 🔄 ह्यूमन-इन-द-लूप म्हणजे काय?

**ह्यूमन-इन-द-लूप (HITL)** हा एक डिझाइन पॅटर्न आहे जिथे AI एजंट्स मानवी इनपुट मागण्यासाठी अंमलबजावणी थांबवतात. हे खालील साठी आवश्यक आहे:

- ✅ **महत्त्वाचे निर्णय** - महत्त्वाची कृती करण्याअगोदर मानवी मंजुरी मिळवा
- ✅ **अनिश्चित परिस्थिती** - AI अनिश्चित असताना मानवी स्पष्टता घ्या
- ✅ **वापरकर्ता पसंती** - वापरकर्त्यांना अनेक पर्यायांमधून निवडायला विचारा
- ✅ **अनुपालन आणि सुरक्षा** - नियमनांसाठी मानवी देखरेख सुनिश्चित करा
- ✅ **परस्परसंवादी अनुभव** - वापरकर्त्याच्या इनपुटला प्रतिसाद देणारे संभाषणात्मक एजंट तयार करा

## 🏗️ मायक्रोसॉफ्ट एजंट फ्रेमवर्कमध्ये कसे कार्य करते

फ्रेमवर्क HITL साठी तीन मुख्य घटक प्रदान करतो:

1. **`RequestInfoExecutor`** - एक विशेष एक्झिक्युटर जो वर्कफ्लो थांबवतो आणि `RequestInfoEvent` पाठवतो
2. **`RequestInfoMessage`** - मानवीकडे पाठविलेल्या टाइप केलेल्या विनंती पेलोडसाठी बेस क्लास
3. **`RequestResponse`** - `request_id` वापरून मानवी प्रतिसाद आणि मूळ विनंत्या यांचे संबंध जोडतो

**वर्कफ्लो पॅटर्न:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 आमचे उदाहरण: वापरकर्त्याच्या पुष्टीसह हॉटेल बुकिंग

आम्ही पर्यायी गंतव्ये सुचवण्याअगोदर मानवी पुष्टी जोडून सशर्त वर्कफ्लो तयार करू:

1. वापरकर्ता गंतव्याची विनंती करतो (उदा., "पॅरिस")
2. `availability_agent` तपासतो की खोल्या उपलब्ध आहेत की नाहीत
3. **जर खोल्या उपलब्ध नाहीत** → `confirmation_agent` विचारतो "आपण पर्याय पाहू इच्छिता का?"
4. वर्कफ्लो `RequestInfoExecutor` वापरून **थांबतो**
5. **मानव प्रतिसाद देतो** "होय" किंवा "नाही" कन्सोल इनपुटद्वारे
6. `decision_manager` प्रतिसादानुसार मार्गदर्शन करतो:
   - **होय** → पर्यायी गंतव्ये दाखवा
   - **नाही** → बुकिंग विनंती रद्द करा
7. अंतिम परिणाम दर्शवा

हे दाखवते की वापरकर्त्यांना एजंटच्या सूचनांवर नियंत्रण कसे देता येते!

---

चला सुरू करूया! 🚀


## टप्पा 1: आवश्यक लायब्ररीस आयात करा

आपण स्टँडर्ड एजंट फ्रेमवर्क घटकांसह **माणूस-इन-द-लूप विशिष्ट वर्ग** आयात करतो:
- `RequestInfoExecutor` - एक्झिक्युटर जो माणसाच्या इनपुटसाठी वर्कफ्लो थांबवतो
- `RequestInfoEvent` - घटना जेव्हा माणसाकडून इनपुट मागितला जातो तेव्हा उत्सर्जित
- `RequestInfoMessage` - टाइप केलेल्या विनंती पेलोडसाठी बेस वर्ग
- `RequestResponse` - माणसाच्या प्रतिसादांना विनंत्यांशी संबंधित करते
- `WorkflowOutputEvent` - वर्कफ्लो आउटपुट शोधण्यासाठी घटना


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## टप्पा 2: संरचित आऊटपुटसाठी Pydantic मॉडेल्स परिभाषित करा

हे मॉडेल्स एजंट्स परत करणारे **स्कीमा** परिभाषित करतात. आम्ही सशर्त वर्कफ्लोमधील सर्व मॉडेल्स ठेवतो आणि पुढील गोष्टी जोडतो:

**मनुष्य-इन-दी-लूपसाठी नवीन:**
- `HumanFeedbackRequest` - `RequestInfoMessage` चा सबक्लास जो माणसांना पाठवले जाणारे विनंती पेलोड परिभाषित करतो
  - यात `prompt` (विचारण्यासाठी प्रश्न) आणि `destination` (उपलब्ध नसलेल्या शहराबद्दल संदर्भ) असतो


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## टप्पा 3: हॉटेल बुकिंग साधन तयार करा

सशर्त कार्यप्रवाहातीलच साधन - हे तपासते की गंतव्यस्थानी खोली उपलब्ध आहेत की नाहीत.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## टप्पा 4: राउटिंगसाठी स्थिती कार्ये परिभाषित करा

आपल्या मानव-इन-द-लूप कार्यप्रवाहासाठी **चार स्थिती कार्ये** आवश्यक आहेत:

**स्थितीय कार्यप्रवाहातून:**
1. `has_availability_condition` - जेव्हा हॉटेल्स उपलब्ध असतात तेव्हा मार्गदर्शित करते
2. `no_availability_condition` - जेव्हा हॉटेल्स उपलब्ध नसतात तेव्हा मार्गदर्शित करते

**मानव-इन-द-लूपसाठी नवीन:**
3. `user_wants_alternatives_condition` - जेव्हा वापरकर्ता पर्यायासाठी "होय" म्हणतो तेव्हा मार्गदर्शित करते
4. `user_declines_alternatives_condition` - जेव्हा वापरकर्ता पर्यायासाठी "नाही" म्हणतो तेव्हा मार्गदर्शित करते


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## पायरी 5: निर्णय व्यवस्थापक अंमलबजावणीकर्ता तयार करा

हे **मानव-इन-द-लूप पॅटर्नचे मुख्य भाग** आहे! `DecisionManager` हा एक सानुकूल `Executor` आहे जो:

1. **`RequestResponse` ऑब्जेक्ट्सच्या माध्यमातून मानवी अभिप्राय घेतो**
2. **वापरकर्त्याचा निर्णय प्रक्रियेत आणतो** (होय/नाही)
3. **संदेश योग्य एजंट्सकडे पाठवून वर्कफ्लो मार्गदर्शन करतो**

मुख्य वैशिष्ट्ये:
- `@handler` डेकॉरेटर वापरून पद्धतींना वर्कफ्लो स्टेप्स म्हणून एक्सपोज करतो
- मूळ विनंती आणि वापरकर्त्याचे उत्तर असलेले `RequestResponse[HumanFeedbackRequest, str]` प्राप्त करतो
- साधे "होय" किंवा "नाही" संदेश देतो जे आमच्या अटी फंक्शन्सला ट्रिगर करतात


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## टप्पा 6: सानुकूल प्रदर्शन कार्यान्वयक तयार करा

सशर्त कार्यप्रवाहातूनच तेच प्रदर्शन कार्यान्वयक - अंतिम परिणाम कार्यप्रवाह आउटपुट म्हणून देते.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## पाउल 7: पर्यावरणीय चलांची लोड करा

LLM क्लायंट (Microsoft Foundry / Azure OpenAI) कॉन्फिगर करा.


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## पाऊल 8: AI एजंट्स आणि एक्झिक्युटर्स तयार करा

आम्ही **सहा कार्यप्रवाह घटक** तयार करतो:

**एजंट्स (AgentExecutor मध्ये वेरapped):**
1. **availability_agent** - टूल वापरून हॉटेल उपलब्धता तपासतो
2. **confirmation_agent** - 🆕 मानवी पुष्टी विनंती तयार करतो
3. **alternative_agent** - पर्यायी शहरे सुचवतो (जेव्हा वापरकर्ता हो म्हणतो)
4. **booking_agent** - बुकिंगला प्रोत्साहन देतो (जेव्हा खोल्या उपलब्ध असतात)
5. **cancellation_agent** - 🆕 रद्दी करणारी संदेश हाताळतो (जेव्हा वापरकर्ता नाही म्हणतो)

**विशेष एक्झिक्युटर्स:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` जे कार्यप्रवाह मानवी इनपुटसाठी थांबवतो
7. **decision_manager** - 🆕 सानुकूल एक्झिक्युटर जे मानवी प्रतिसादावर आधारित मार्गदर्शन करतो (आधीच वरिभागी व्याख्यित)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## पाऊल 9: मानव-इन-द-लूपसह वर्कफ्लो तयार करा

आता आपण **शर्तीपूर्ण मार्गदर्शन** सह वर्कफ्लो ग्राफ तयार करतो ज्यात मानव-इन-द-लूप मार्ग समाविष्ट आहे:

**वर्कफ्लो रचना:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**महत्वाचे कड्या:**
- `availability_agent → confirmation_agent` (जेव्हा खोली नाहीत)
- `confirmation_agent → prepare_human_request` (ट्रान्सफॉर्म प्रकार)
- `prepare_human_request → request_info_executor` (मानवासाठी थांबा)
- `request_info_executor → decision_manager` (नेहमी - RequestResponse पुरवतो)
- `decision_manager → alternative_agent` (जेव्हा वापरकर्ता "हो" म्हणतो)
- `decision_manager → cancellation_agent` (जेव्हा वापरकर्ता "नाही" म्हणतो)
- `availability_agent → booking_agent` (जेव्हा खोली उपलब्ध असतात)
- सर्व मार्ग `display_result` वर समाप्त होतात


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## पाऊल 10: टेस्ट केस 1 चालवा - उपलब्धता नसलेले शहर (मानवी पुष्टीसह पॅरिस)

हा चाचणी **पूर्ण मानवी-इन-लूप चक्र** दर्शवितो:

1. पॅरिसमधील हॉटेलची विनंती करा
2. availability_agent तपासतो → खोल्या नसल्याचे
3. confirmation_agent मानवी-सामना करणारा प्रश्न तयार करतो
4. request_info_executor **कार्यप्रवाह थांबवतो** आणि `RequestInfoEvent` उत्सर्जित करतो
5. **अ‍ॅप्लिकेशन घटना ओळखते आणि कन्सोलमध्ये वापरकर्त्याला विचारते**
6. वापरकर्ता "होय" किंवा "नाही" टाइप करतो
7. अ‍ॅप्लिकेशन प्रतिक्रिया `send_responses_streaming()` द्वारे पाठविते
8. decision_manager प्रतिसादावर आधारित मार्गक्रमण करतो
9. अंतिम निकाल प्रदर्शित होतो

**महत्त्वाचा नमुना:**
- प्रथम आवृत्तीकरिता `workflow.run_stream()` वापरा
- पुढील आवृत्त्यांसाठी `workflow.send_responses_streaming(pending_responses)` वापरा
- मानवी इनपुट आवश्यक असल्यास `RequestInfoEvent` ऐका
- अंतिम निकाल पकडण्यासाठी `WorkflowOutputEvent` ऐका


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## चरण 11: चाचणी प्रकरण 2 चालवा - उपलब्धतासह शहर (स्टॉकहोम - मानवी इनपुट आवश्यक नाही)

ही चाचणी **प्रत्यक्ष मार्ग** दर्शवते जेव्हा खोल्या उपलब्ध असतात:

1. स्टॉकहोममध्ये हॉटेलची विनंती करा
2. availability_agent तपासतो → खोल्या उपलब्ध आहेत ✅
3. booking_agent बुकिंग सुचवतो
4. display_result पुष्टी दाखवते
5. **कोणताही मानवी इनपुट आवश्यक नाही!**

जेव्हा खोल्या उपलब्ध असतात तेव्हा कार्यप्रवाह पूर्णपणे मानवी-इन-द-लूप मार्गाला वगळतो.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## मुख्य मुद्दे आणि मानवी-इन-द-लूप सर्वोत्तम प्रथाः

### ✅ तुम्ही काय शिकलात:

#### 1. **RequestInfoExecutor पॅटर्न**
Microsoft Agent Framework मधील मानवी-इन-द-लूप पॅटर्न तीन मुख्य घटकांचा वापर करतो:
- `RequestInfoExecutor` - वर्कफ्लो थांबवतो आणि इव्हेंट्स उत्सर्जित करतो
- `RequestInfoMessage` - टाइप केलेल्या विनंती पेलोडसाठी बेस वर्ग (याचा सबक्लास करा!)
- `RequestResponse` - मानवी प्रतिसादांना मूळ विनंत्यांसोबत जोडतो

**महत्त्वाचे समजः**
- `RequestInfoExecutor` स्वतः इनपुट गोळा करत नाही - ते फक्त वर्कफ्लो थांबवते
- तुमच्या अॅप्लिकेशन कोडने `RequestInfoEvent` ऐकले पाहिजे आणि इनपुट गोळा करावा
- तुम्ही `send_responses_streaming()` कॉल करा ज्यात `request_id` ते वापरकर्त्याच्या उत्तराचे मॅपिंग असावे

#### 2. **स्ट्रीमिंग अंमलबजावणी पॅटर्न**
```python
# पहिली पुनरावृत्ती
stream = workflow.run_stream(initial_request)

# पुढील पुनरावृत्त्या (मानवी इनपुटनंतर)
stream = workflow.send_responses_streaming(pending_responses)

# नेहमी घटना प्रक्रिया करा
events = [event async for event in stream]
```

#### 3. **इव्हेंट-ड्रिव्हन आर्किटेक्चर**
वर्कफ्लो नियंत्रित करण्यासाठी विशिष्ट इव्हेंट्स ऐका:
- `RequestInfoEvent` - मानवी इनपुट आवश्यक (वर्कफ्लो थांबलेला)
- `WorkflowOutputEvent` - अंतिम निकाल उपलब्ध (वर्कफ्लो पूर्ण)
- `WorkflowStatusEvent` - स्थिती बदल (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, इत्यादी)

#### 4. **@handler सह सानुकूल एक्झिक्युटर्स**
`DecisionManager` दाखवते कसे:
- `@handler` डेकोरेटरचा वापर करून पद्धती वर्कफ्लो स्टेप्स म्हणून एक्स्पोझ करा
- टाइप केलेली मेसेजेस प्राप्त करा (उदा. `RequestResponse[HumanFeedbackRequest, str]`)
- इतर एक्झिक्युटर्सना मेसेजेस पाठवून वर्कफ्लो मार्गदर्शित करा
- `WorkflowContext` द्वारे संदर्भ प्राप्त करा

#### 5. **मानवी निर्णयांसह सशर्त मार्गदर्शन**
तुम्ही मानवी प्रतिसादांचे मूल्यमापन करणाऱ्या अटीचे फंक्शन्स तयार करू शकता:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 प्रत्यक्ष जगातील अनुप्रयोगः

1. **मंजुरी वर्कफ्लोज़**
   - खर्च अहवाल प्रक्रियेसाठी व्यवस्थापकांचे मंजुरी घ्या
   - स्वयंचलित ईमेल पाठवण्यापूर्वी मानवी पुनरावलोकन आवश्यक आहे
   - उच्च मूल्य व्यवहारांच्या पुष्टीपूर्वक व्यवहार करा

2. **सामग्री नियंत्रण**
   - शंकास्पद सामग्री मानवी पुनरावलोकनासाठी फ्लॅग करा
   - एज केसवर अंतिम निर्णयासाठी मॉडरेटर विचारणे
   - AI आत्मविश्वास कमी असताना मानवीकडे वाढवा

3. **ग्राहक सेवा**
   - सामान्य प्रश्नांसाठी AI स्वयंचलित उत्तर द्या
   - गुंतागुंतीच्या समस्यांसाठी मानवी एजंटकडे वाढवा
   - ग्राहकाला विचार करा की त्यांना मानवीशी बोलायचे आहे का

4. **डेटा प्रक्रिया**
   - अस्पष्ट डेटा नोंदी निपटण्यासाठी मानवी मदत मागा
   - अस्पष्ट दस्तऐवजांचे AI अर्थ पुष्टी करा
   - वापरकर्त्यांना अनेक वैध अर्थांमधून निवडू द्या

5. **सुरक्षा-गंभीर प्रणाली**
   - अपरिवर्तनीय क्रियांसाठी मानवी पुष्टी आवश्यक
   - संवेदनशील डेटावर प्रवेश करण्यासाठी मंजुरी घ्या
   - नियमन उद्योगांमध्ये निर्णयांची पुष्टी करा (आरोग्यसेवा, वित्त)

6. **अंतःक्रियात्मक एजंट्स**
   - पुढील प्रश्न विचारणारे संभाषणात्मक बॉट्स तयार करा
   - वापरकर्त्यांना गुंतागुंतीतील प्रक्रियांत मार्गदर्शन करणारे विजार्ड तयार करा
   - मानवींसह टप्प्याटप्प्याने सहकार्यासाठी एजंट्स डिझाइन करा

### 🔄 तुलना: मानवी-इन-द-लूप सह आणि न कसा आहे

| वैशिष्ट्य | सशर्त वर्कफ्लो | मानवी-इन-द-लूप वर्कफ्लो |
|---------|---------------------|---------------------------|
| **अंमलबजावणी** | एकल `workflow.run()` | `run_stream()` + `send_responses_streaming()` सह लूप |
| **वापरकर्ता इनपुट** | नाही (पूर्णपणे स्वयंचलित) | `input()` किंवा UI द्वारे संवादात्मक प्रम्प्ट्स |
| **घटक** | एजंट्स + एक्झिक्युटर्स | + RequestInfoExecutor + DecisionManager |
| **इव्हेंट्स** | फक्त AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, इत्यादी |
| **थांबवणे** | थांबवणे नाही | RequestInfoExecutor वर वर्कफ्लो थांबतो |
| **मानवी नियंत्रण** | मानवी नियंत्रण नाही | मानवरांकडून महत्त्वाचे निर्णय घेतले जातात |
| **उपयोग प्रकरण** | ऑटोमेशन | सहकार्य आणि देखरेख |

### 🚀 प्रगत पॅटर्न्स:

#### अनेक मानवी निर्णय बिंदू
आपण एका वर्कफ्लोमध्ये अनेक `RequestInfoExecutor` नोड्स ठेवू शकता:
```python
.add_edge(agent1, request_info_1)  # पहिला मानवी निर्णय
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # दुसरा मानवी निर्णय
.add_edge(decision_manager_2, final_agent)
```

#### टाइमआउट हाताळणी
मानवी प्रतिसादांसाठी टाइमआउट लागू करा:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # सुरक्षित पर्यायावर डीफॉल्ट करा
```

#### समृद्ध UI एकत्रीकरण
`input()` ऐवजी वेब UI, Slack, Teams इत्यादींसह एकत्रित करा:
```python
if isinstance(event, RequestInfoEvent):
    # वापरकर्त्याच्या पसंतीच्या चॅनेलवर सूचना पाठवा
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### सशर्त मानवी-इन-द-लूप
विशिष्ट परिस्थितींमध्येच मानवी इनपुट मागा:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # आत्मविश्वास कमी असेल किंवा मूल्य जास्त असेल तरच माणसाकडे मार्गा
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ सर्वोत्तम प्रथा:

1. **नेहमी RequestInfoMessage चे सबक्लास करा**
   - टाईप सुरक्षितता आणि प्रमाणीकरण प्रदान करते
   - UI रेंडरिंगसाठी समृद्ध संदर्भ सक्षम करते
   - प्रत्येक विनंती प्रकाराचा हेतू स्पष्ट करतो

2. **स्पष्टीकरणात्मक प्रम्प्ट वापरा**
   - तुम्ही काय विचारत आहात याचा संदर्भ समाविष्ट करा
   - प्रत्येक निवडीच्या परिणामाचे स्पष्टीकरण द्या
   - प्रश्न सोपे आणि स्पष्ट ठेवा

3. **अनपेक्षित इनपुट हाताळा**
   - वापरकर्त्यांच्या प्रतिसादांची पडताळणी करा
   - अवैध इनपुटसाठी पूर्वनिर्धारित किंमती द्या
   - स्पष्ट त्रुटी संदेश द्या

4. **विनंती आयडींचे ट्रॅक ठेवा**
   - request_id आणि प्रतिसादांच्या सहसंबंधाचा वापर करा
   - हाताने स्थिती व्यवस्थापित करू नका

5. **नॉन-ब्लॉकिंग डिझाइन करा**
   - इनपुटची प्रतिक्षा करताना थ्रेड्स ब्लॉक करू नका
   - संपूर्णपणे_async_ पॅटर्न्स वापरा
   - एकाच वेळी अनेक वर्कफ्लो उदाहरणांना समर्थन द्या

### 📚 संबंधित संकल्पना:

- **एजंट मिडलवेअर** - एजंट कॉल्सवर इंटरसेप्ट करा (पूर्वीचा नोटबुक)
- **वर्कफ्लो स्टेट मॅनेजमेंट** - रनदरम्यान वर्कफ्लो स्थिती टिकवा
- **मल्टी-एजंट सहकार** - मानवी-इन-द-लूप एजंट संघांसह एकत्र करा
- **इव्हेंट-ड्रिव्हन आर्किटेक्चर्स** - इव्हेंटसह प्रतिसादात्मक प्रणाली तयार करा

---

### 🎓 अभिनंदन!

तुम्ही Microsoft Agent Framework सह मानवी-इन-द-लूप वर्कफ्लोमध्ये पारंगत झाला आहात! तुम्हाला आता कसे माहित आहे:
- ✅ मानवी इनपुट गोळा करण्यासाठी वर्कफ्लो थांबवायचे
- ✅ RequestInfoExecutor आणि RequestInfoMessage वापरायचे
- ✅ इव्हेंटसह स्ट्रीमिंग अंमलबजावणी हाताळायची
- ✅ @handler सह सानुकूल एक्झिक्युटर्स तयार करायचे
- ✅ मानवी निर्णयांवर आधारित वर्कफ्लोज मार्गदर्शित करायचे
- ✅ मानवी सहकार्याने अंतःक्रियात्मक AI एजंट्स तयार करायचे

**हे विश्वासार्ह, नियंत्रणात ठेवता येणाऱ्या AI प्रणालींसाठी एक महत्त्वाचा पॅटर्न आहे!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
